# Inference

Inference on my SFT model on raw data

In [13]:
import torch

if torch.cuda.is_available():
    device = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device)
    
    total_memory = torch.cuda.get_device_properties(device).total_memory
    allocated_memory = torch.cuda.memory_allocated(device)
    reserved_memory = torch.cuda.memory_reserved(device)
    free_memory = total_memory - reserved_memory
    
    print(f"GPU: {gpu_name}")
    print(f"Total Memory:     {total_memory / 1e6:.2f} MB | {total_memory / 1e9:.2f} GB")
    print(f"Allocated Memory: {allocated_memory / 1e6:.2f} MB | {allocated_memory / 1e9:.2f} GB")
    print(f"Reserved Memory:  {reserved_memory / 1e6:.2f} MB | {reserved_memory / 1e9:.2f} GB")
    print(f"Free Memory:      {free_memory / 1e6:.2f} MB | {free_memory / 1e9:.2f} GB")
else:
    print("No CUDA GPU available")

GPU: NVIDIA A30
Total Memory:     25338.05 MB | 25.34 GB
Allocated Memory: 0.00 MB | 0.00 GB
Reserved Memory:  0.00 MB | 0.00 GB
Free Memory:      25338.05 MB | 25.34 GB


In [31]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

BASE_MODEL_NAME = "Qwen/Qwen3.5-2B"
MODEL_NAME = "checkpoint_models/checkpoint-9597"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto")
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, device_map="auto")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [16]:
print(f"[INFO] Model on device: {model.device}")
print(f"[INFO] Model using dtype: {model.dtype}")

[INFO] Model on device: cuda:0
[INFO] Model using dtype: torch.bfloat16


In [35]:
messages = [
    {"role": "user", "content": "我在看要不要做Eurostar去阿姆斯特丹\n我算了之後\n更便宜\n也比較方便\n只是要做四小時"},
]

tokenized_chat = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, return_tensors="pt")
tokenized_chat

'<|im_start|>user\n我在看要不要做Eurostar去阿姆斯特丹\n我算了之後\n更便宜\n也比較方便\n只是要做四小時<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

In [36]:

inputs = tok(tokenized_chat, return_tensors="pt")
streamer = TextStreamer(tok)

print("[INFO] Fine-tuned model output:")
_ = model.generate(**inputs, streamer=streamer)

print("\n[INFO] Base model output:")
_ = base_model.generate(**inputs, streamer=streamer)

[INFO] Fine-tuned model output:
<|im_start|>user
我在看要不要做Eurostar去阿姆斯特丹
我算了之後
更便宜
也比較方便
只是要做四小時<|im_end|>
<|im_start|>assistant
<think>

</think>



/vol/bitbucket/lc3223/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:2636: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


對對
寶貝好累。
你還要吗
愛你
快.202


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.



[INFO] Base model output:
<|im_start|>user
我在看要不要做Eurostar去阿姆斯特丹
我算了之後
更便宜
也比較方便
只是要做四小時<|im_end|>
<|im_start|>assistant
<think>

</think>

你的計算非常準確！從倫敦出發，**Eurostar 去阿姆斯特丹**確實是目前最
